# Video Dialogue Locator — CPU vs GPU ASR benchmark

This notebook is a standalone dev tool, not part of the shipped pipeline
(see `AGENTS.md` / `prompt.txt` in the main repo for why: a hosted GPU
notebook was the agreed way to test GPU speedups without owning a GPU,
kept out of the pipeline's own architecture).

**What this actually measures:** the video-dialogue-locator pipeline has
two independent bottlenecks — downloading the video (network-bound) and
transcribing it with faster-whisper (compute-bound, ASR). **A GPU only
affects the second one.** It has zero effect on download speed. So this
notebook downloads the video *once*, then times the *same* transcription
step twice — once forced onto CPU, once on the Colab GPU — so the
download time (and any network variance) cancels out of the comparison.

**Before running:** in Colab, go to `Runtime -> Change runtime type` and
pick a GPU (e.g. T4), or this whole comparison is moot.

**Steps:**
1. Confirm a GPU is attached
2. Get the project source onto this machine
3. Install system + Python dependencies
4. Download the target video once, extract audio once
5. Transcribe on CPU, timed
6. Transcribe on GPU, timed
7. Compare
8. (Optional) run the full `vdl locate` CLI end-to-end on GPU

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

If that errored or shows no devices: `Runtime -> Change runtime type ->
Hardware accelerator -> GPU`, then re-run this cell before continuing.

## 2. Get the project source

Two ways to get the code onto this Colab machine — use whichever applies:

- **Option A (public repo, or you have a token):** edit `REPO_URL` below
  and run the git-clone cell. For a private repo, use a URL of the form
  `https://<token>@github.com/<owner>/<repo>.git` (generate a fine-scoped
  [personal access token](https://github.com/settings/tokens) — don't
  paste a real token into a notebook you intend to share or commit).
- **Option B (no public URL / no token handy):** skip the clone cell and
  run the upload cell instead — zip the repo locally
  (`zip -r vdl.zip . -x '.venv/*' '.vdl_cache/*' 'outputs/*' '.git/*'`)
  and upload it via the file picker that appears.

In [ ]:
# Option A: clone from GitHub. Edit this if your remote differs, or if
# the repo is private (see markdown above for the token URL form).
REPO_URL = "https://github.com/sharonprabhu11/Video-Dialogue-Locator.git"

!git clone "$REPO_URL" vdl_repo
%cd vdl_repo

In [ ]:
# Option B: only run this instead of the clone cell above if you don't
# have a git URL handy. Upload the zip described in the markdown cell.
# from google.colab import files
# import zipfile, os
# uploaded = files.upload()
# zip_name = next(iter(uploaded))
# with zipfile.ZipFile(zip_name) as zf:
#     zf.extractall("vdl_repo")
# os.chdir("vdl_repo")

## 3. Install dependencies

Colab already ships CUDA + a compatible NVIDIA driver, so no extra CUDA
setup is needed — `faster-whisper`'s prebuilt wheels include GPU support.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -e ".[dev]"
!ffmpeg -version | head -1
!python -c "import ctranslate2; print('ctranslate2', ctranslate2.__version__, '- CUDA devices:', ctranslate2.get_cuda_device_count())"

The last line should report at least 1 CUDA device. If it reports 0, the
GPU runtime isn't actually attached — go back to step 1.

## 4. Download the video once, extract audio once

Neither of these steps involves the GPU — doing them once and reusing the
result for both benchmark runs is what keeps the CPU-vs-GPU comparison
fair (otherwise you'd be measuring network variance, not compute).

In [ ]:
from pathlib import Path
from vdl import acquisition, audio as vdl_audio

# Edit these for whatever video/moment you want to benchmark against.
URL = "https://ok.ru/video/248244667877"

workdir = Path("/content/vdl_bench")
video = acquisition.acquire_video(URL, workdir / "video", cache_dir=workdir / "cache")
audio_asset = vdl_audio.extract_audio(video, workdir / "audio")
print(f"video: {video.local_path} ({video.duration_s:.1f}s, {video.fps:.2f}fps)")
print(f"audio: {audio_asset.path}")

## 5-6. Transcribe on CPU, then on GPU — same audio, same model

In [ ]:
import time
from vdl import transcription

ASR_MODEL = "small"  # match whatever you'd actually run locally

t0 = time.time()
cpu_transcript = transcription.transcribe(audio_asset, ASR_MODEL, device="cpu")
cpu_seconds = time.time() - t0
print(f"CPU transcription: {cpu_seconds:.1f}s ({len(cpu_transcript.segments)} segments)")

In [ ]:
t0 = time.time()
gpu_transcript = transcription.transcribe(audio_asset, ASR_MODEL, device="cuda")
gpu_seconds = time.time() - t0
print(f"GPU transcription: {gpu_seconds:.1f}s ({len(gpu_transcript.segments)} segments)")

## 7. Compare

In [ ]:
speedup = cpu_seconds / gpu_seconds if gpu_seconds else float("nan")
print(f"audio duration : {audio_asset.duration_s:.1f}s")
print(f"CPU (int8)     : {cpu_seconds:.1f}s")
print(f"GPU (float16)  : {gpu_seconds:.1f}s")
print(f"speedup        : {speedup:.1f}x")

same_text = cpu_transcript.segments[0].text.strip() == gpu_transcript.segments[0].text.strip() if cpu_transcript.segments and gpu_transcript.segments else None
print(f"first segment matches: {same_text}  (sanity check — CPU/GPU can differ slightly due to precision, not a bug)")

## 8. (Optional) run the full pipeline end-to-end on GPU

This re-downloads nothing (the acquisition cache from step 4 covers it)
and gives you the actual CLI output — timestamp, frame, extracted image —
using the GPU for the ASR stage.

In [ ]:
TARGET_TEXT = "My mind rebels at stagnation"

!vdl locate --url "$URL" --text "$TARGET_TEXT" --asr-device cuda --cache-dir /content/vdl_bench/cache --out-dir /content/vdl_bench/outputs

In [ ]:
from IPython.display import Image, display
import glob

frames = sorted(glob.glob("/content/vdl_bench/outputs/*.png"))
if frames:
    display(Image(filename=frames[-1]))
else:
    print("no frame written — check the CLI output above for the actual status")